In [28]:
import sys
import os

current_file_path = os.path.abspath(os.getcwd()) # Funciona solo para JupyterNotebooks

# Obtener la ruta del directorio raíz del proyecto
project_root = os.path.abspath(os.path.join(os.path.dirname(current_file_path)))

# Añadir el directorio raíz del proyecto al sys.path
sys.path.append(project_root)

from config import INPUTS_FOLDER

statement_path = os.path.join(INPUTS_FOLDER, 'test_files', 'Nu', 'Nu_debit_statement.pdf')

In [29]:
from document_reader import PDFReader
from bank_detector import DefaultBankDetector
import logging
logging.getLogger('pdfminer').setLevel(logging.ERROR) # Abvoid PDFMiner warnings

bank_detector = DefaultBankDetector(PDFReader(statement_path))

extracted_words = bank_detector.extracted_words
extracted_words

,page,text,x0,top,x1,bottom
0,1,Mario,399.800,44.910,426.110,54.910
1,1,Alexis,428.760,44.910,456.370,54.910
2,1,Canudas,459.020,44.910,500.110,54.910
3,1,Zertuche,502.760,44.910,545.000,54.910
4,1,Cuenta,427.310,59.910,461.160,69.910
...,...,...,...,...,...,...
2145,9,099,342.736,803.528,358.104,811.528
2146,9,1133,360.224,803.528,376.272,811.528
2147,9,9,521.280,815.528,526.240,823.528
2148,9,de,528.360,815.528,537.920,823.528


In [30]:
bank = bank_detector.detect_bank()
statement_type = bank_detector.detect_statement_type()
statement_properties = bank_detector.get_statement_properties()

print(f'{bank} - {statement_type}')

nu - debit


In [31]:
from table_boundary_detector import TransactionTableBoundaryDetector

boundary_detector = TransactionTableBoundaryDetector(extracted_words, statement_properties)

start_idx = boundary_detector.start_idx
end_idx = boundary_detector.end_idx

if statement_type == 'credit' and (start_idx is None or end_idx is None):
    print('New format')
    statement_properties = bank_detector.get_statement_properties(new_credit_format= True)
    boundary_detector = TransactionTableBoundaryDetector(extracted_words, statement_properties)
    
    start_idx = boundary_detector.start_idx
    end_idx = boundary_detector.end_idx

df_table = boundary_detector.get_filtered_table_words()

print(f'{start_idx} - {end_idx}')
df_table

149 - 1336


,page,text,x0,top,x1,bottom
0,2,FECHA,58.00,110.84,90.46,120.84
1,2,DEL,137.84,110.84,155.81,120.84
2,2,01,158.46,110.84,169.26,120.84
3,2,AL,171.91,110.84,184.00,120.84
4,2,30,186.65,110.84,199.50,120.84
...,...,...,...,...,...,...
1182,5,Cajita:,180.95,583.00,210.62,593.00
1183,5,Mi,213.27,583.00,224.38,593.00
1184,5,Primera,227.03,583.00,263.09,593.00
1185,5,Cajita,265.74,583.00,292.49,593.00


In [32]:
from row_segmenter import TransactionRowSegmenter

row_segmenter = TransactionRowSegmenter(df_table, statement_properties)

column_delimitation = row_segmenter.delimit_column_positions()
grouped_rows = row_segmenter.group_rows()

print(row_segmenter.row_threshold)
print(column_delimitation)
grouped_rows

13.888739583333333
{'columns': ['FECHA', '(DE) (\\d{2}) (ENE|FEB|MAR|ABR|MAY|JUN|JUL|AGO|SEP|OCT|NOV|DIC) (A) (\\d{2}) (ENE|FEB|MAR|ABR|MAY|JUN|JUL|AGO|SEP|OCT|NOV|DIC) (\\(\\d{2}) (DÍAS\\))', 'MONTO EN PESOS MEXICANOS'], 'x0': [58.0, None, 394.63000000000034], 'x1': [90.46, None, 544.4500000000004]}


,row_group,text,words,top,bottom,page
0,0,FECHA DEL 01 AL 30 SEP 2024 (30 DÍAS) MONTO EN...,"[(FECHA, 58.0, 90.46), (DEL, 137.84, 155.81), ...",110.840,120.840,2
1,1,Depósito en Cajita: Mi Primera Cajita 29 SEP 2...,"[(Depósito, 135.84, 177.73), (en, 180.38, 192....",131.930,142.930,2
2,2,29 SEP 2024 +$700.00 MARIO ALEXIS CANUDAS ZERT...,"[(29, 56.0, 67.74), (SEP, 70.39, 88.1300000000...",158.020,169.520,2
3,3,"Depósito SPEI, Hora: 13:55:15, Recibido de BBV...","[(Depósito, 135.84, 173.54100000000003), (SPEI...",179.109,242.109,2
4,4,CHICHI SUAREZ MERIDA Compra 29 SEP 2024 -$64.50,"[(CHICHI, 135.84, 170.58), (SUAREZ, 173.230000...",256.390,267.390,2
...,...,...,...,...,...,...
72,72,Depósito en Cajita: Mi Primera Cajita 05 SEP 2...,"[(Depósito, 135.84, 177.73), (en, 180.38, 192....",406.950,417.950,5
73,73,MARIO ANTONIO CANUDAS AREVALO TRANSFERENCIA A ...,"[(MARIO, 135.84, 168.3), (ANTONIO, 170.95, 216...",433.040,458.040,5
74,74,"Depósito SPEI, Hora: 17:22:05, Recibido de SAN...","[(Depósito, 135.84, 173.54100000000003), (SPEI...",466.129,542.629,5
75,75,Pago a tu tarjeta de crédito Nu 03 SEP 2024 -$...,"[(Pago, 135.84, 159.35999999999999), (a, 162.0...",556.910,567.910,5


In [33]:
from table_reconstructor import TransactionTableReconstructor

table_reconstructor = TransactionTableReconstructor(grouped_rows, column_delimitation, statement_properties)

table_reconstructor.get_structured_table()

,FECHA,(DE) (\d{2}) (ENE|FEB|MAR|ABR|MAY|JUN|JUL|AGO|SEP|OCT|NOV|DIC) (A) (\d{2}) (ENE|FEB|MAR|ABR|MAY|JUN|JUL|AGO|SEP|OCT|NOV|DIC) (\(\d{2}) (DÍAS\)),MONTO EN PESOS MEXICANOS
0,FECHA,DEL 01 AL 30 SEP 2024 (30 DÍAS) MONTO EN PESOS...,
1,29 SEP 2024,Depósito en Cajita: Mi Primera Cajita,-$735.50
2,29 SEP 2024,MARIO ALEXIS CANUDAS ZERTUCHE consulta,+$700.00
3,,"Depósito SPEI, Hora: 13:55:15, Recibido de BBV...",
4,29 SEP 2024,CHICHI SUAREZ MERIDA Compra,-$64.50
...,...,...,...
72,05 SEP 2024,Depósito en Cajita: Mi Primera Cajita 2024,"-$1,200.51"
73,05 SEP 2024,MARIO ANTONIO CANUDAS AREVALO TRANSFERENCIA A ...,"+$1,200.00"
74,,"Depósito SPEI, Hora: 17:22:05, Recibido de SAN...",
75,03 SEP 2024,Pago a tu tarjeta de crédito Nu 2024,"-$2,377.49"


In [34]:
df_structured = table_reconstructor.reconstruct_table()
df_structured

,FECHA,(DE) (\d{2}) (ENE|FEB|MAR|ABR|MAY|JUN|JUL|AGO|SEP|OCT|NOV|DIC) (A) (\d{2}) (ENE|FEB|MAR|ABR|MAY|JUN|JUL|AGO|SEP|OCT|NOV|DIC) (\(\d{2}) (DÍAS\)),MONTO EN PESOS MEXICANOS
0,29 SEP 2024,Depósito en Cajita: Mi Primera Cajita,-$735.50
1,29 SEP 2024,MARIO ALEXIS CANUDAS ZERTUCHE consulta,+$700.00
2,29 SEP 2024,CHICHI SUAREZ MERIDA Compra,-$64.50
3,29 SEP 2024,Retiro de Cajita: Mi Primera Cajita,+$100.00
4,27 SEP 2024,Depósito en Cajita: Mi Primera Cajita,"-$1,090.00"
5,27 SEP 2024,MARIA FERNANDA CANUDAS ZERTUCHE Medicina,+$200.00
6,27 SEP 2024,FARMACIAS SIMILARES Compra,-$200.00
7,27 SEP 2024,Retiro de Cajita: Mi Primera Cajita,"+$1,000.00"
8,26 SEP 2024,PARRILLA GRAN PLAZA Compra,-$110.00
9,26 SEP 2024,Retiro de Cajita: Mi Primera Cajita,+$200.00


In [35]:
from table_normalizer import TransactionTableNormalizer

normalizer = TransactionTableNormalizer(df_structured, extracted_words, statement_properties)

df_normalized = normalizer.normalize_table()
df_normalized

,Date,Description,Amount,Type
0,2024-09-29,Depósito en Cajita: Mi Primera Cajita,-735.50,Cargo
1,2024-09-29,MARIO ALEXIS CANUDAS ZERTUCHE consulta,700.00,Abono
2,2024-09-29,CHICHI SUAREZ MERIDA Compra,-64.50,Cargo
3,2024-09-29,Retiro de Cajita: Mi Primera Cajita,100.00,Abono
4,2024-09-27,Depósito en Cajita: Mi Primera Cajita,-1090.00,Cargo
5,2024-09-27,MARIA FERNANDA CANUDAS ZERTUCHE Medicina,200.00,Abono
6,2024-09-27,FARMACIAS SIMILARES Compra,-200.00,Cargo
7,2024-09-27,Retiro de Cajita: Mi Primera Cajita,1000.00,Abono
8,2024-09-26,PARRILLA GRAN PLAZA Compra,-110.00,Cargo
9,2024-09-26,Retiro de Cajita: Mi Primera Cajita,200.00,Abono


In [36]:
from data_exporter import CsvExporter
from config import OUTPUTS_FOLDER

exporter = CsvExporter(df_normalized)
file_path = os.path.join(OUTPUTS_FOLDER, bank, f'{bank}_{statement_type}.csv')

exporter.export_to_csv(file_path)

Data exported to c:\Users\mario\Proyectos\Aether\Aether_v1\documents\outputs\nu\nu_debit.csv successfully.
